# QAI PR 4 – Quantum Error Simulation and Measurement Error Mitigation

# QAI Practical

This notebook recreates the practical shown in the supplied PDF. Run cells from top to bottom in Google Colab or a local Jupyter environment. The install cell is intended for Colab/Jupyter.

This reproduces the noisy 3-qubit circuit, creates all 8 computational-basis calibration circuits, builds a calibration matrix, and applies matrix-based mitigation.

### 1. Install packages

Installs Qiskit, Qiskit Aer, and Matplotlib. The Aer noise simulator is needed to model gate and readout errors.

In [ ]:
!pip -q install qiskit qiskit-aer matplotlib


### 2. Import noise and circuit tools

Imports circuit construction and transpilation, the Aer simulator, noise-model classes, histogram plotting, NumPy for matrix operations, and Matplotlib.

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error, ReadoutError
from qiskit.visualization import plot_histogram
import numpy as np
import matplotlib.pyplot as plt

print("Libraries imported successfully.")


### 3. Build the ideal circuit

Creates a three-qubit circuit, applies H to qubit 0 and two CX gates to correlate the three qubits, then measures all qubits. This is the circuit that will be executed with a noise model.

In [ ]:
qc = QuantumCircuit(3, 3)

# Apply gates
qc.h(0)
qc.cx(0, 1)
qc.cx(0, 2)

# Measure all qubits
qc.measure([0, 1, 2], [0, 1, 2])

print("Quantum circuit created successfully.")
print(qc.draw())


### 4. Define the noise model

Adds depolarizing error probabilities to H and CX gates. It also defines a readout-error matrix: each measurement has a 3% chance of being reported as the opposite bit. These are illustrative noise settings.

In [ ]:
noise_model = NoiseModel()

# Depolarizing error for single-qubit and two-qubit gates
error_1 = depolarizing_error(0.01, 1)
error_2 = depolarizing_error(0.02, 2)

noise_model.add_all_qubit_quantum_error(error_1, ['h'])
noise_model.add_all_qubit_quantum_error(error_2, ['cx'])

# Measurement/readout error
readout_error = ReadoutError([
    [0.97, 0.03],
    [0.03, 0.97]
])
noise_model.add_all_qubit_readout_error(readout_error)

print("Noise model created successfully.")


### 5. Run noisy circuit

Creates an Aer simulator using the noise model, transpiles the circuit for that simulator, and executes 1000 shots. The counts represent noisy observed outcomes.

In [ ]:
simulator = AerSimulator(noise_model=noise_model)
compiled_qc = transpile(qc, simulator)

job = simulator.run(compiled_qc, shots=1000)
result = job.result()
counts = result.get_counts()

print("Original noisy counts:")
print(counts)


### 6. Create calibration circuits

Builds all eight three-bit computational-basis states (000 through 111). Each circuit prepares one basis state and measures it, allowing the readout behavior to be estimated.

In [ ]:
# Create calibration circuits for all 2^3 computational basis states.
n = 3
calibration_circuits = []

for state in range(2**n):
    label = format(state, f'0{n}b')
    cal = QuantumCircuit(n, n)

    # Prepare the computational-basis state.
    for qubit, bit in enumerate(reversed(label)):
        if bit == '1':
            cal.x(qubit)

    cal.measure(range(n), range(n))
    calibration_circuits.append(cal)

print("Calibration circuits created:", len(calibration_circuits))


### 7. Run calibration experiments

Executes every calibration circuit using the same noisy simulator and stores its measurement counts. Each result estimates how a prepared state is observed under the readout noise.

In [ ]:
calibration_results = []

for cal in calibration_circuits:
    compiled_cal = transpile(cal, simulator)
    cal_job = simulator.run(compiled_cal, shots=1000)
    result_cal = cal_job.result()
    calibration_results.append(result_cal.get_counts())

print("Calibration completed successfully.")


### 8. Build calibration matrix

Creates an 8×8 matrix. Each column corresponds to a prepared basis state and each row to a measured state; entries are normalized frequencies from the calibration runs.

In [ ]:
states = [format(i, '03b') for i in range(2**n)]
cal_matrix = np.zeros((2**n, 2**n))

for column, counts_cal in enumerate(calibration_results):
    total = sum(counts_cal.values())
    for row, state in enumerate(states):
        cal_matrix[row, column] = counts_cal.get(state, 0) / total

print("Calibration matrix:")
print(np.round(cal_matrix, 3))


### 9. Mitigate measurement probabilities

Converts the observed counts to probabilities, applies the pseudoinverse of the calibration matrix to estimate corrected probabilities, clips small negative numerical values, normalizes, and converts the estimates back to counts.

In [ ]:
# Convert measured counts to a probability vector.
measured_probs = np.array([
    counts.get(state, 0) / 1000
    for state in states
])

# Calculate corrected probabilities using the inverse/pseudoinverse.
mitigated_probs = np.linalg.pinv(cal_matrix) @ measured_probs

# Remove small negative numerical values and normalize.
mitigated_probs = np.maximum(mitigated_probs, 0)
if mitigated_probs.sum() == 0:
    raise ValueError("Mitigation produced zero total probability.")
mitigated_probs = mitigated_probs / mitigated_probs.sum()

# Convert probabilities back to counts.
mitigated_counts = {
    state: int(round(prob * 1000))
    for state, prob in zip(states, mitigated_probs)
    if prob > 0
}

print("Mitigated counts:")
print(mitigated_counts)


### 10. Compare original and mitigated counts

Prints both dictionaries so the effect of the calibration-based correction can be compared numerically.

In [ ]:
print("Original counts:")
print(counts)

print("\\nMitigated counts:")
print(mitigated_counts)


### 11. Plot comparison

Displays original noisy counts and mitigated counts side by side. Differences illustrate how readout calibration can reduce measurement bias; mitigation is an estimate, not a guarantee of perfect results.

In [ ]:
fig = plot_histogram(
    [counts, mitigated_counts],
    legend=['Original', 'Mitigated'],
    figsize=(7, 4)
)
plt.show()
